# 🎤 EVEZ Vocaloid — Custom Singing Voice Model
Free. No license. Your voice, your songs.

Uses **RVC v2** (Retrieval-based Voice Conversion) — the same tech behind AI covers.
Takes a 1-3 minute voice sample → trains a model → sings anything.

## What you need:
- 1-3 min of you speaking or singing (phone recording is fine)
- A melody/midi or reference vocal track to convert
- 5-10 min on free Colab T4

## Output:
- `.pth` voice model file (portable, works in RVC/WebUI)
- `.index` feature index for quality
- Can sing in any language, any style

In [ ]:
# Cell 1: Install RVC
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
%cd Retrieval-based-Voice-Conversion-WebUI
!pip install -r requirements.txt
!pip install torch torchaudio soundfile pydub
print('✅ RVC installed')

In [ ]:
# Cell 2: Upload your voice dataset (1-3 min of clean speech/singing)
from google.colab import files
import os

print('Upload your voice sample (1-3 minutes, WAV preferred):')
uploaded = files.upload()
sample_file = list(uploaded.keys())[0]
print(f'✅ Uploaded: {sample_file}')

# Create dataset directory structure
!mkdir -p /content/dataset/raw
!cp {sample_file} /content/dataset/raw/

# Slice long audio into 10-second chunks for training
from pydub import AudioSegment
audio = AudioSegment.from_file(sample_file)
chunk_len = 10000  # 10 seconds
os.makedirs('/content/dataset/sliced', exist_ok=True)
for i, start in enumerate(range(0, len(audio), chunk_len)):
    chunk = audio[start:start+chunk_len]
    chunk.export(f'/content/dataset/sliced/{i:04d}.wav', format='wav')

n_chunks = len(os.listdir('/content/dataset/sliced'))
print(f'✅ Sliced into {n_chunks} chunks')

In [ ]:
# Cell 3: Extract features + train RVC model
# This is the core training step — ~5-10 min on T4

MODEL_NAME = 'EVEZ_Voice_v1'
EXPERIMENT_DIR = f'/content/RVC/models/{MODEL_NAME}'
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Step 3a: Process dataset and extract features
print('Step 1/3: Processing audio...')
!python Retrieval-based-Voice-Conversion-WebUI/infer/modules/vc/trainset_preprocess_pipeline_print.py \
  /content/dataset/sliced 40000 4 {EXPERIMENT_DIR} True

print('Step 2/3: Extracting features...')
!python Retrieval-based-Voice-Conversion-WebUI/infer/modules/vc/extract_f0_print.py \
  {EXPERIMENT_DIR} 4 crepe

!python Retrieval-based-Voice-Conversion-WebUI/infer/modules/vc/extract_feature_print.py \
  {EXPERIMENT_DIR} v2 4

print('Step 3/3: Training RVC model...')
# Train for 200 epochs (good quality, ~5 min on T4)
!python Retrieval-based-Voice-Conversion-WebUI/infer/modules/vc/train.py \
  v2 {EXPERIMENT_DIR} 40 0 crepe True 200 0 1 0

print(f'✅ Model trained: {EXPERIMENT_DIR}')

In [ ]:
# Cell 4: Build feature index
!python Retrieval-based-Voice-Conversion-WebUI/infer/modules/vc/train_index.py \
  {EXPERIMENT_DIR} v2
print('✅ Feature index built')

In [ ]:
# Cell 5: Inference — make your voice SING
# Upload a reference vocal or instrumental track
print('Upload a song/instrumental to convert to your voice:')
uploaded = files.upload()
song_file = list(uploaded.keys())[0]
print(f'✅ Uploaded: {song_file}')

# Run RVC inference
import torch
from infer.modules.vc.pipeline import Pipeline

OUTPUT_PATH = '/content/evez_vocaloid_output.wav'

# Find the trained model
model_path = f'{EXPERIMENT_DIR}/v2/model.pth'
index_path = f'{EXPERIMENT_DIR}/v2/added_IVF512_Flat_nprobe_1_v2.index'

print(f'Model: {model_path}')
print(f'Index: {index_path}')
print('Generating vocal...')

# Use RVC's CLI for inference
!python Retrieval-based-Voice-Conversion-WebUI/infer-cli.py \
  -m {model_path} \
  -i {song_file} \
  -o {OUTPUT_PATH} \
  -t 0 \
  -s {MODEL_NAME} \
  -k crepe \
  -r 0 \
  -f0 0

print(f'✅ Output: {OUTPUT_PATH}')

In [ ]:
# Cell 6: Download everything
from google.colab import files
import glob

# Download model + index + output
for pattern in [model_path, index_path, OUTPUT_PATH]:
    if os.path.exists(pattern):
        files.download(pattern)
        print(f'⬇️ {pattern}')

print('\n🎤 EVEZ Vocaloid complete!')
print('Your .pth model works in:')
  - RVC WebUI (local or Colab)')
  - OpenUtau (free Vocaloid editor)')
  - Any RVC-compatible tool')

---
## 🎵 Using Your Voice Model

### Option A: RVC WebUI (Colab, free)
Run the RVC WebUI on Colab, upload your .pth model, convert any song.

### Option B: OpenUtau (local, free)
1. Download OpenUtau: https://github.com/stakira/OpenUtau
2. Place your .pth in the singers folder
3. Compose with piano roll, your voice sings it

### Option C: API endpoint
Host the model on a free GPU tier (Kaggle/RunPod free tier)
and call it from the EVEZ Provider Gateway.

### Voice style presets:
| Style | Transpose | Pitch | Description |
|-------|-----------|-------|-------------|
| Natural | 0 | crepe | Your voice as-is |
| High vocal | +5 | crepe | Brighter, pop vocal range |
| Deep growl | -7 | crepe | Lower, heavier |
| Robot | 0 | dio | Mechanized, glitchy |